# Vigenere - evaluation

Results for the trained PATE model: the decoding grid under two conditions,
the latency comparison between prediction and exhaustive search, and the
out-of-distribution check.

Requires a trained checkpoint in `checkpoints/vigenere_pate`.

In [ ]:
import os, sys
from pathlib import Path

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = Path('/content/drive/MyDrive/cryptanalysis-ai')
    %pip install -q pytorch-lightning torchmetrics matplotlib
else:
    REPO_DIR = Path.cwd()
    if REPO_DIR.name == 'notebooks':
        REPO_DIR = REPO_DIR.parent

os.environ['CRYPT_DATA_DIR'] = str(REPO_DIR / 'data')
os.environ['CRYPT_CKPT_DIR'] = str(REPO_DIR / 'checkpoints')
os.environ['CRYPT_LOG_DIR']  = str(REPO_DIR / 'tb_logs')
sys.path.insert(0, str(REPO_DIR / 'src'))
print('repo:', REPO_DIR)


In [ ]:
import time
import numpy as np
import torch, config
from models.vigenere.model_pate import VigenereSolver, UNKNOWN_KEY
from models.vigenere import inference as vinf, evaluation as ev

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ckpt = sorted((config.CHECKPOINT_DIR / "vigenere_pate").glob("*best-acc*.ckpt"))[-1]
model = VigenereSolver.load_from_checkpoint(str(ckpt), map_location=DEVICE)
model.to(DEVICE).eval(); model.freeze()
print("checkpoint:", ckpt.name)

char2idx, idx2char = vinf.build_vocab(config.ALPHABET)
clean = ev.load_clean_text(config.data_file("validation_set.txt"), config.ALPHABET)
print(f"validation: {len(clean):,} characters")


## Decoding grid

Each cell is measured twice: through the full pipeline, where a low validity
score sends the input to an exhaustive scoring pass, and with that safeguard
disabled so that decryption proceeds from the blind estimate whether or not it
is correct.

In [ ]:
TEST_KEY_LENS = [3, 5, 8, 12, 20, 32]
TEST_LENGTHS  = [128, 256, 512]
N = 500

@torch.no_grad()
def decoding_grid(m):
    out = {}
    for k in TEST_KEY_LENS:
        out[k] = {}
        for L in TEST_LENGTHS:
            c_fb = c_no = tot = key_ok = blind_ok = fb = 0
            for i in range(N):
                s = (i * L * 13) % max(1, len(clean) - L)
                key = ev.get_fixed_key(k, i, config.ALPHABET)
                cipher, plain = vinf.encrypt_vigenere(clean[s:s+L], key, char2idx, idx2char)
                src = vinf.text_to_tensor(cipher, char2idx, DEVICE)

                r = m.decode(src)
                dec = "".join(idx2char.get(x, "") for x in r["plaintext"].tolist())
                n = min(len(plain), len(dec))
                c_fb += sum(a == b for a, b in zip(plain[:n], dec[:n]))
                key_ok += int(r["best_key_len"] == k)
                fb += int(r["method"] == "fallback_bruteforce")

                r0 = m.decode(src, use_fallback=False)
                dec0 = "".join(idx2char.get(x, "") for x in r0["plaintext"].tolist())
                c_no += sum(a == b for a, b in zip(plain[:n], dec0[:n]))
                blind_ok += int(r0["best_key_len"] == k)
                tot += n

            out[k][L] = dict(char=c_fb/max(tot,1)*100, char_nofb=c_no/max(tot,1)*100,
                             key=key_ok/N*100, blind=blind_ok/N*100, fb=fb/N*100)
    return out

grid = decoding_grid(model)
classical = ev.run_classical_eval(clean, TEST_LENGTHS, TEST_KEY_LENS,
                                  config.ALPHABET, num_samples=N,
                                  min_k=model.min_key_len, max_k=model.max_key_len)

print(f"{'k':>3} {'L':>5} | {'CHAR%':>6} {'KEY%':>6} | {'CHAR%':>6} {'BLND%':>6} | {'FB%':>5} | {'CLS':>6}")
print(f"{'':>3} {'':>5} | {'pipeline':>13} | {'no fallback':>13} | {'':>5} | {'CHAR%':>6}")
print("-" * 62)
for k in TEST_KEY_LENS:
    for L in TEST_LENGTHS:
        c = grid[k][L]
        print(f"{k:>3} {L:>5} | {c['char']:>6.1f} {c['key']:>6.1f} | "
              f"{c['char_nofb']:>6.1f} {c['blind']:>6.1f} | {c['fb']:>5.1f} | "
              f"{classical[k][L]['char_acc']:>6.1f}")

cells = [grid[k][L] for k in TEST_KEY_LENS for L in TEST_LENGTHS]
print(f"\nmean pipeline CHAR%    : {np.mean([c['char'] for c in cells]):.1f}")
print(f"mean no-fallback CHAR% : {np.mean([c['char_nofb'] for c in cells]):.1f}")
print(f"mean fallback rate     : {np.mean([c['fb'] for c in cells]):.1f}")
print(f"mean classical CHAR%   : "
      f"{np.mean([classical[k][L]['char_acc'] for k in TEST_KEY_LENS for L in TEST_LENGTHS]):.1f}")


## Latency

Both strategies share the same weights. Direct prediction uses one pass to
estimate the period and one to decrypt under it; the exhaustive variant scores
every candidate length and decrypts each in full. Reported direct-prediction
times include the fallback pass on the inputs where it engages.

In [ ]:
LAT_LENGTHS  = [128, 256, 400, 512]
LAT_KEY_LENS = list(range(model.min_key_len, model.max_key_len + 1))
LAT_N = 200

@torch.no_grad()
def latency(m):
    out = {}
    for L in LAT_LENGTHS:
        ms_d = ms_b = 0.0
        count = 0
        for k in LAT_KEY_LENS:
            for i in range(LAT_N // len(LAT_KEY_LENS) + 1):
                s = (i * L * 13) % max(1, len(clean) - L)
                key = ev.get_fixed_key(k, i, config.ALPHABET)
                cipher, _ = vinf.encrypt_vigenere(clean[s:s+L], key, char2idx, idx2char)
                src = vinf.text_to_tensor(cipher, char2idx, DEVICE)
                t0 = time.perf_counter(); m.decode(src); ms_d += (time.perf_counter()-t0)*1000
                t0 = time.perf_counter(); m.decode_bruteforce(src); ms_b += (time.perf_counter()-t0)*1000
                count += 1
        out[L] = (ms_d/count, ms_b/count)
    return out

lat = latency(model)
for L, (d, b) in lat.items():
    print(f"L={L:>4}: direct {d:>5.1f} ms | exhaustive {b:>5.1f} ms")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(LAT_LENGTHS, [lat[L][0] for L in LAT_LENGTHS], "o-",
        color="#2563EB", lw=2, ms=7, label="direct prediction")
ax.plot(LAT_LENGTHS, [lat[L][1] for L in LAT_LENGTHS], "o-",
        color="#DC2626", lw=2, ms=7, label="exhaustive search")
ax.set_xlabel("text length (chars)")
ax.set_ylabel("average latency (ms)")
ax.set_title("Latency: search vs prediction")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(config.STORAGE_ROOT / "latency.png", dpi=150, bbox_inches="tight")
plt.show()


## Out-of-distribution check

Place a Turkish corpus of a different register at `data/ood_text.txt`. The
character-frequency distance is reported alongside the results: if it is near
zero the two corpora share a distribution and the comparison says nothing.

In [ ]:
res = ev.run_ood_eval(
    model,
    ood_path=config.data_file("ood_text.txt"),
    validation_path=config.data_file("validation_set.txt"),
    alphabet_chars=config.ALPHABET,
    test_lengths=(128, 256, 512),
    test_key_lens=[3, 5, 8, 12, 20, 32],
    num_samples=200,
    device=DEVICE,
    unknown_key=UNKNOWN_KEY,
)
ev.print_ood_summary(res)
